In [203]:
#Christian Yanez

#imports
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error as MSE

df = pd.read_csv('./Realestate.csv')#load the file as instructed
#print(df)# The original set of information so I can compare it to the result
#standardization
#There are 6 X's
#there are 414 entries
X_features = df.columns[df.columns.str.startswith("X")]#This is to specially get the X features if the name of starts with X
#print(df[X_features].std())
#print(df[X_features].mean())
df[X_features] = (df[X_features] - df[X_features].mean()) / df[X_features].std()#(Xi - mean)/std
df.to_csv("standardized.csv", index=False)

print(df)


      No  X1 transaction date  X2 house age  \
0      1             2012.917          32.0   
1      2             2012.917          19.5   
2      3             2013.583          13.3   
3      4             2013.500          13.3   
4      5             2012.833           5.0   
..   ...                  ...           ...   
409  410             2013.000          13.7   
410  411             2012.667           5.6   
411  412             2013.250          18.8   
412  413             2013.000           8.1   
413  414             2013.500           6.5   

     X3 distance to the nearest MRT station  X4 number of convenience stores  \
0                                  84.87882                               10   
1                                 306.59470                                9   
2                                 561.98450                                5   
3                                 561.98450                                5   
4                                 3

In [190]:
#Scikit-learn baseline 
#This is a line of best fit, so I need to pair x with y for this function
#The Root Mean Square Error is the square root of the Mean Square Error as shown in the lecture
#There are functions within the sklearn library to get these values without needing to calculate them

X = df[X_features]
Y = df["Y house price of unit area"]
LR = LinearRegression()#we imported sklearn in the first block
LR.fit(X,Y)

#now we make a preidction
Y_prediction = LR.predict(X)
RMSE = MSE(Y, Y_prediction) ** (1/2)
print("The Root Mean Squared Error is: ", RMSE)

The Root Mean Squared Error is:  8.78246646460277


In [191]:
#Batch gradient descent implementation

#to use the Numpy Linear Regression, as explained in the lecture videos, it is a class.

#The RMSE from the previous question was 8.78, and we want to multiply that amount by 1.1 to set the limit.

class Linear_Regression():
    def __init__(self, learning_rate, iterations):
        self.learning_rate = learning_rate
        self.iterations = iterations

    def fit(self, X, Y):
        self.m, self.n = X.shape#m is the amount of rows, amount of columns

        self.W = np.zeros(self.n)
        self.b = 0
        self.X = X
        self.Y = Y

        #this uses the entire batch for iterations
        for i in range(self.iterations):
            self.update_weights()
        return self
        
    def update_weights(self):
        Y_prediction = self.predict(self.X)
        dW = -(2*self.X.T).dot(self.Y-Y_prediction)/self.m
        db = -2 * np.sum(self.Y-Y_prediction)/self.m
        self.W = self.W - self.learning_rate*dW
        self.b = self.b - self.learning_rate*db
        return self

    def predict(self, X):
        return X.dot(self.W) + self.b

np_LR = Linear_Regression(0.1, 100)#these will be the learning rate and iterations
np_LR.fit(X, Y)
Y_prediction = np_LR.predict(X)

rmse = np.sqrt(np.mean((Y - Y_prediction) ** 2))#use the same equation from before

print("RMSE:", rmse)

RMSE: 8.782706161047614


In [192]:
#Stochastic Batch Gradient
#The main difference is that Stochastic Batch Gradient uses a much smaller set of data for each iteration.
#I take the same class from the previous question and I edit it to make it take smaller samples
#I plan on doing this by using a for loop

class Linear_Regression():
    def __init__(self, learning_rate, iterations):
        self.learning_rate = learning_rate
        self.iterations = iterations

    def fit(self, X, Y):
        self.m, self.n = X.shape

        self.W = np.zeros(self.n)
        self.b = 0
        self.X = X
        self.Y = Y

        for i in range(self.iterations):#this also updates as well
            for j in range(self.m):#lets say for example, I want this to look at each individual entry for each iteration
                self.individual_x = self.X.iloc[j]#without iloc, it would get me a syntax error
                self.individual_y = self.Y.iloc[j]
                self.update_weights()
        return self

    def update_weights(self):
        Y_prediction = self.predict(self.individual_x)#refactor the update_weights so I can take the individual entries instead

        dW = -2 * self.individual_x * (self.individual_y - Y_prediction)
        db = -2 * (self.individual_y - Y_prediction)

        self.W = self.W - self.learning_rate * dW
        self.b = self.b - self.learning_rate * db

        return self
                

    def predict(self, X):
        return X.dot(self.W) + self.b

np_SGD = Linear_Regression(0.001, 100)#I realized that I needed to lower the learning rate.

np_SGD.fit(X, Y)

Y_prediction = np_SGD.predict(X)

rmse = np.sqrt(np.mean((Y - Y_prediction) ** 2))

print("RMSE:", rmse)

RMSE: 8.783887023770259


In [193]:
#question 2

df = pd.read_csv('./data.csv')

#prior probability
x1_count = 0
x0_count = 0
X_features = df.columns[df.columns.str.startswith("X")]#I borrowed this line from the previous question so I can single out the columns
X = df[X_features]
m, n = X.shape
for i in range(m):
    if X.iloc[i, 0] == 1:
        x1_count += 1

    elif X.iloc[i, 0] == 0:
        x0_count += 1
x0_prob = x0_count/m
x1_prob = x1_count/m
print("probability of x=0: ", x0_prob)
print("probability of x=1: ", x1_prob)

y1_count = 0
y0_count = 0
Y_features = df.columns[df.columns.str.startswith("Y")]
Y = df[Y_features]
r, s = Y.shape
for i in range(r):
    if Y.iloc[i, 0] == 1:
        y1_count += 1

    elif Y.iloc[i, 0] == 0:
        y0_count += 1
y0_prob = y0_count/r
y1_prob = y1_count/r
print("probability of y=0: ", y0_prob)
print("probability of y=1: ", y1_prob)


probability of x=0:  0.5555555555555556
probability of x=1:  0.4444444444444444
probability of y=0:  0.5252525252525253
probability of y=1:  0.47474747474747475


In [194]:
#calculate the likelihoods of the following probabilities
#Likelihood is calculated by taking the probability of both events happening, not only one of them
#Then divded by the probability of the occurance on the right side of the "|"
x0y0 = 0
x1y0 = 0
x0y1 = 0
x1y1 = 0

for i in range(m):
    if (X.iloc[i, 0] == 0) & (Y.iloc[i,0] == 0):
        x0y0 += 1

    elif (X.iloc[i, 0] == 1) & (Y.iloc[i,0] == 0):
        x1y0 += 1
        
    elif (X.iloc[i, 0] == 0) & (Y.iloc[i,0] == 1):
        x0y1 += 1
        
    elif (X.iloc[i, 0] == 1) & (Y.iloc[i,0] == 1):
        x1y1 += 1
        
x0_given_y0 = (x0y0/m)/(y0_prob)#calculate the probability of X and Y being specific values within the entire set before dividing
x1_given_y0 = (x1y0/m)/(y0_prob)#that by the probability of just Y being that specific value.
x0_given_y1 = (x0y1/m)/(y1_prob)
x1_given_y1 = (x1y1/m)/(y1_prob)

print("x=0 | y=0 probability is: ", x0_given_y0)
print("x=1 | y=0 probability is:", x1_given_y0)
print("x=0 | y=1 probability is:", x0_given_y1)
print("x=1 | y=1 probability is:", x1_given_y1)

x=0 | y=0 probability is:  0.5769230769230769
x=1 | y=0 probability is: 0.423076923076923
x=0 | y=1 probability is: 0.5319148936170213
x=1 | y=1 probability is: 0.4680851063829787


In [195]:
#Bayes theorem.
#P(A|B) = (P(B|A) * P(A))/(P(B))
#in this case, A = y and B = x
#use the values from the previous questions, it's just assembly to get the final answer
y1_given_x0 = x0_given_y1 * y1_prob / x0_prob
y0_given_x0 = x0_given_y0 * y0_prob / x0_prob
y1_given_x1 = x1_given_y1 * y1_prob / x1_prob
y0_given_x1 = x1_given_y0 * y0_prob / x1_prob

print("y=1 | x=0 probability is:", y1_given_x0)
print("y=0 | x=0 probability is:", y0_given_x0)
print("y=1 | x=1 probability is:", y1_given_x1)
print("y=0 | x=1 probability is:", y0_given_x1)

y=1 | x=0 probability is: 0.45454545454545453
y=0 | x=0 probability is: 0.5454545454545454
y=1 | x=1 probability is: 0.5
y=0 | x=1 probability is: 0.5


In [196]:
#Question 3
#between class scatter matrix

from sklearn import datasets

#the original matrix has 100 rows with 4 columns
X, y = datasets.make_blobs(n_samples=100, n_features=4, centers=2,
                           cluster_std=1.5, random_state=123)
y = np.array([-1 if i == 0 else 1 for i in y])

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
#the original matrix has 100 rows with 4 columns
#The SB matrix is the sum of the (amount of entries * the difference between the class mean and the overall mean)
def between_class_scatter(X, y):
    overall_mean = X.mean(axis=0)

    SB = np.zeros((X.shape[1], X.shape[1]))

    for c in np.unique(y):
        Xc = X[y == c]#this is to check the label if it is -1 or 1.
        class_mean = Xc.mean(axis=0)
        nc = Xc.shape[0]

        mean_diff = class_mean - overall_mean#difference between the class mean and mean of the whole set

        SB += nc * np.outer(mean_diff, mean_diff)#summation of the values

    return SB
SB = between_class_scatter(X_train, y_train)#insert the values of the training data into this function
print(SB)


[[ 1.42177386e+00 -1.48370842e+01 -7.83733634e+01 -1.38981884e+01]
 [-1.48370842e+01  1.54834094e+02  8.17874222e+02  1.45036139e+02]
 [-7.83733634e+01  8.17874222e+02  4.32022577e+03  7.66118860e+02]
 [-1.38981884e+01  1.45036139e+02  7.66118860e+02  1.35858203e+02]]


In [197]:
#Within-class scatter matrices ans Sw^-1

def within_class_scatter(X, y):
    within_class_matrices = {}#create a new matrix to store the class matrices

    # Total within-class scatter
    SW = np.zeros((X.shape[1], X.shape[1]))#similar setup to the last question where we create a zero matrix

    for c in np.unique(y):
        Xc = X[y == c]
        class_mean = Xc.mean(axis=0)
        SWC = np.zeros((X.shape[1], X.shape[1]))

        for x in Xc:
            difference = x - class_mean#but instead, we take the difference between the individual elements and the average of the class
            SWC += np.outer(difference, difference)

        within_class_matrices[c] = SWC#adding the current class matrix to the collection of class matrices
        SW += SWC#the second summation

    # Inverse of Sw
    SWinv = np.linalg.inv(SW)#inverse the matric

    return within_class_scatter, SW, SWinv

within_class_scatter, SW, SWinv = within_class_scatter(X_train, y_train)#initialize the values that were returned from the function

print("Within-class scatter for each class:")
for c, matrix in within_class_matrices.items():
    print("Class", c, "\n", matrix)

print("SW: \n", SW)

print("Inverse of Sw: \n", SWinv)

Within-class scatter for each class:
Class -1 
 [[106.64405638  10.57121868 -12.1814986   14.60184927]
 [ 10.57121868 119.16367913  -5.72990248  10.47724786]
 [-12.1814986   -5.72990248  86.05322087  15.7115349 ]
 [ 14.60184927  10.47724786  15.7115349  101.52648328]]
Class 1 
 [[ 82.46926179  -4.04261776  -4.23894557  -6.4196867 ]
 [ -4.04261776  84.86055693   3.72403296  -4.61228307]
 [ -4.23894557   3.72403296  68.56560078 -13.16579016]
 [ -6.4196867   -4.61228307 -13.16579016  46.20319683]]
SW: 
 [[189.11331817   6.52860091 -16.42044417   8.18216257]
 [  6.52860091 204.02423606  -2.00586952   5.86496479]
 [-16.42044417  -2.00586952 154.61882164   2.54574474]
 [  8.18216257   5.86496479   2.54574474 147.72968011]]
Inverse of Sw: 
 [[ 5.35588997e-03 -1.57131974e-04  5.71698050e-04 -3.00255075e-04]
 [-1.57131974e-04  4.91228014e-03  5.01215342e-05 -1.87181528e-04]
 [ 5.71698050e-04  5.01215342e-05  6.53128940e-03 -1.46204087e-04]
 [-3.00255075e-04 -1.87181528e-04 -1.46204087e-04  6.79

In [198]:
#Eigenvalues, eigenvectors, and projection directions 

#numpy has a build in eigenvalue function that we can use on the product of Sw^-1 and SB

productSWinvSB = SWinv @ SB #I don't want an outer product for this, the notation is dot product.

print("SW^-1*SB \n", productSWinvSB)
eigenvalues, eigenvectors = np.linalg.eig(productSWinvSB)

print("\nEigenvectors: ")#remember that eigenvectors are stored as columns python
print(eigenvectors)

for i in range(len(eigenvectors)):
    current_eigenvector = eigenvectors[:,i]#due to eigenvectors being columns, [i] would not work in this loop
    projection = X_train @ current_eigenvector

    # Classify using a threshold of 0
    y_pred = np.where(projection >= 0, 1, -1)

    print("Current Eigenvector: \n", current_eigenvector)

    print("\nProjection: \n", projection)


largest_index = np.argmax(eigenvalues)#the best eigenvalue is always the largest, so grab the value by simply using argmax

best_eigenvalue = eigenvalues[largest_index]#eigenvalues are a 1d row
best_eigenvector = eigenvectors[:, largest_index]#But Eigenvectors are natrually stored as a column hence the [:,i] to reference it properly

print("\nBest eigenvalue:\n", best_eigenvalue)

print("\nBest eigenvector:\n", best_eigenvector)

# Project training data onto the best eigenvector
projection = X_train @ best_eigenvector

print("\nProjection:\n", projection)
#The best eigenvector is usually the largest one because it has the most variance between classes
#This can be easily found by using np.argmax which returns the index that the largest value in the set
#was located.

SW^-1*SB 
 [[-0.03068665  0.32023408  1.69156027  0.29996956]
 [-0.07443403  0.77676485  4.1030753   0.7276109 ]
 [-0.50977798  5.31984655 28.10082227  4.98320483]
 [-0.08063909  0.84151852  4.44512116  0.78826694]]

Eigenvectors: 
[[-0.99984555  0.05874635 -0.29976091 -0.56773754]
 [-0.00250869  0.14249608  0.65004626 -0.4911584 ]
 [-0.01718134  0.97591602 -0.24448846 -0.03431092]
 [-0.00271783  0.15437502  0.65407083  0.65974258]]
Current Eigenvector: 
 [-0.99984555 -0.00250869 -0.01718134 -0.00271783]

Projection: 
 [-5.45894862 -2.05152676 -2.99249263 -3.2485215  -2.2346525  -5.68934132
 -4.10290673 -3.40784801 -5.89376608 -5.31021005 -2.58992178 -0.21499629
 -6.10891994 -3.22096845 -4.33313665 -4.41056504 -4.51149501 -4.55187652
 -4.09723886 -5.22017913 -4.82029527 -2.31047087 -6.0828985  -2.30890057
 -4.33549164 -5.46130234 -5.9168248  -1.71594699 -4.0441274  -3.38456447
 -3.22523637 -6.54904801 -4.34578754 -5.41859601 -4.05124088 -6.61263179
 -3.90562607 -4.4424793  -4.04667991 

In [199]:
 #Train accuracy with ideal direction
from sklearn.metrics import accuracy_score

projection = X_train @ best_eigenvector

# 1D classification
y_pred = np.where(projection >= 0, 1, -1)#designate that the classes get labeled properly, without needed to use if and elif statements.

accuracy = accuracy_score(y_train, y_pred) #Which sees if y_train matches up with y_pred

print("Best Eigenvector: \n",best_eigenvector)

print("\nProjection: \n",projection)

print("\nPredicted labels: \n", y_pred)

print("\nAccuracy:\n", accuracy)

Best Eigenvector: 
 [0.05874635 0.14249608 0.97591602 0.15437502]

Projection: 
 [10.49344401  8.23403937 -6.23778243 -5.08140446 -6.72768436 10.93272746
 -6.64475407 10.29143476 -6.90063926 -7.48667905  9.20802779 -4.20466442
 -5.59896819  7.66837953 -3.54573059 10.25743448 -5.49893221 -4.2407423
 -5.27944813 -5.34346907 11.11043834 -4.91767919 -6.66868634 -6.90746514
  9.60594471  7.52798134  7.2455161   9.15383918 -4.6228364  -8.92768934
 -1.94799768  8.9249959  12.15341543  9.43776759 -3.08870353  9.56730677
 -3.61019215 -4.89304479  8.39122099 -3.51991109  9.41591469  8.87569777
 11.22517511 -4.29068917 -3.95598145  8.67028721 10.60203636 11.52335963
 -5.18338783 -7.30538879 10.33219259 -5.94491653 -6.18164718 -5.80082771
 -7.14751497 11.99170186  9.88351589 10.12535624 -6.28384107  8.9039398
 12.81448959 10.37107446 10.56376386 -4.60320796  9.58593198 -5.26765969
 -3.30152767  7.96083662 -7.36595707 -5.11734733 -4.96162237 11.23722769
  9.08314793 -2.78198007 -3.59642149 -4.97050

In [200]:
#Other eigenvector comparisons
for i in range(len(eigenvectors)):
    if i == largest_index:#simply just repeat the previous code and don't run the best eigenvector, results will then appear as normal
        continue
    current_eigenvector = eigenvectors[:,i]#due to eigenvectors being columns, [i] would not work in this loop
    projection = X_train @ current_eigenvector

    # 1D classification
    y_pred = np.where(projection >= 0, 1, -1)#

    # Calculate accuracy
    accuracy = accuracy_score(y_train, y_pred)

    print("Current Eigenvector: \n", current_eigenvector)

    print("\nProjection: \n", projection)

    print("\nPredicted labels: \n", y_pred)

    print("\nAccuracy: \n", accuracy)
#As shown here, the accuracies reported are around the ~.48-.53 range. Nowhere near the previous parts response.
#The best accuracy was reported to have an accuracy of 1.0.

Current Eigenvector: 
 [-0.99984555 -0.00250869 -0.01718134 -0.00271783]

Projection: 
 [-5.45894862 -2.05152676 -2.99249263 -3.2485215  -2.2346525  -5.68934132
 -4.10290673 -3.40784801 -5.89376608 -5.31021005 -2.58992178 -0.21499629
 -6.10891994 -3.22096845 -4.33313665 -4.41056504 -4.51149501 -4.55187652
 -4.09723886 -5.22017913 -4.82029527 -2.31047087 -6.0828985  -2.30890057
 -4.33549164 -5.46130234 -5.9168248  -1.71594699 -4.0441274  -3.38456447
 -3.22523637 -6.54904801 -4.34578754 -5.41859601 -4.05124088 -6.61263179
 -3.90562607 -4.4424793  -4.04667991 -1.92392302 -2.73369031 -2.70849627
 -4.39982842 -2.59439201 -5.35935697 -3.91690761 -7.18376622 -2.76912088
 -5.55671492 -3.86425293 -4.00310872 -6.30851767 -5.17492316 -4.96265452
 -2.48114208 -5.54805999 -3.8225815  -5.89432644 -3.1733268  -4.49044494
 -1.72762181 -4.94125734 -3.51488316 -5.74236007 -1.92295264 -7.12910303
 -7.45087048 -3.38373112 -2.39295043 -4.40304458 -4.90448869 -2.97176354
 -2.26961864 -2.77834815 -2.84427785

In [201]:
#Test accuracy and scikit-learn LDA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis#provided code
lda = LinearDiscriminantAnalysis(n_components=1)

projection = X_test @ best_eigenvector#use the test information for the testing accuracy this time

# Using the same process as before but swap prediction with the test
y_pred = np.where(projection >= 0, 1, -1)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred) 

#now we use the LDA to compare
lda.fit(X_train, y_train)

# Predict the test data
y_test_pred_lda = lda.predict(X_test)

# Calculate test accuracy
test_accuracy_lda = accuracy_score(y_test, y_test_pred_lda)

print("Test labels: \n", y_pred)

print("\nAccuracy for X_test data: \n" ,accuracy)

print("\nLDA Test Predicted labels: \n", y_test_pred_lda)

print("\nLDA Test Accuracy: \n", test_accuracy_lda)

#the accuracies of the best eigenvector and the LDA are equal at 1.0.

Test labels: 
 [-1  1  1  1  1 -1 -1  1  1 -1  1 -1  1 -1  1  1  1 -1  1 -1]

Accuracy for X_test data: 
 1.0

LDA Test Predicted labels: 
 [-1  1  1  1  1 -1 -1  1  1 -1  1 -1  1 -1  1  1  1 -1  1 -1]

LDA Test Accuracy: 
 1.0
